# KNOWLEDGE_GRAPH FOR RAG
# TRADITIONAL RAG:
# Phase 1: Indexing
1.Data Ingestion: Source documents (PDFs, text, web pages) are gathered and parsed.
2.Text Chunking: Large documents are split into smaller, manageable text segments (e.g., paragraphs or sentences) so they fit into context windows.
3.Embedding: These text chunks are converted into numerical vector representations that capture semantic meaning.
4.Indexing: The vector embeddings are stored in a Vector Database, allowing for fast similarity searches

#Phase 2:Retrieval & Generation (At Query Time)
1.User Query: The user submits a question in natural language.
2.Query Embedding: The user's query is converted into a vector using the same embedding model used during Phase 1.
3.Similarity Retrieval: The system compares the query vector against the vector database to fetch the "Top-K" most relevant text chunks.
4.Prompt Augmentation: The retriever aggregates the most relevant text chunks and merges them with the original user query into a structured prompt.
5.Generation: The augmented prompt is fed to the Large Language Model (LLM), which synthesizes a grounded answer based on the provided context.

#How Modern RAG (GraphRAG) Changes the Game

1.From Chunks to Concepts:Modern RAG extracts distinct Entities (e.g., people, products, places) and Relationships from text, mapping out a structured web of knowledge rather than isolated snippets.
2.Relationship-Aware Retrieval: Instead of relying strictly on word similarity, the system can "hop" across connected entities (e.g., Product X -> Manufactured by Y -> Located in Z), allowing it to answer complex, multi-hop queries that would leave traditional RAG stumbling.
3.Global vs. Local Search: Modern systems offer both Local Search (gathering direct neighbors of an entity) and Global Search (traversing the whole graph to summarize massive datasets).
4.Hybrid Search: Modern pipelines (e.g., using Neo4j) combine vector search and graph traversal together, retrieving highly specific text chunks while maintaining an understanding of the overarching context.

*Most powerful is the Hybrid Search="KG+VECTOR SEARCH".

In [1]:
import sys

print(sys.executable)

C:\Users\pauld\anaconda3\envs\kg\python.exe


In [2]:
import torch
print(torch.__version__)

2.12.1+cpu


In [4]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    filename="config.json"
)

'C:\\Users\\pauld\\.cache\\huggingface\\hub\\models--sentence-transformers--all-MiniLM-L6-v2\\snapshots\\1110a243fdf4706b3f48f1d95db1a4f5529b4d41\\config.json'

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    local_files_only=True
)

print("Model Loaded Successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4902.56it/s]

Model Loaded Successfully


In [6]:
documents = [
    "Alice works at Google",
    "Bob works at Amazon",
    "Google is located in California",
    "Amazon is located in Seattle",
    "California is in USA",
    "Seattle is in USA",
    "Alice manages Charlie",
    "Charlie works on Project X",
    "Project X belongs to Google"
]

embeddings = model.encode(documents)

print(embeddings.shape)

(9, 384)


In [7]:
print(embeddings[0])

[-6.54982328e-02 -4.85866740e-02  2.31765285e-02 -8.39683507e-03
 -7.09242597e-02 -1.95771903e-02  3.53872143e-02 -8.79472792e-02
  1.46423168e-02 -1.67159867e-02  4.93690483e-02  3.72428969e-02
  2.11928505e-02 -4.91509624e-02  4.09443788e-02  4.61461954e-02
  7.55951032e-02 -4.47494499e-02 -1.21587561e-02 -8.11670870e-02
 -7.70077705e-02 -3.19758356e-02  6.86445013e-02 -3.40956114e-02
  5.75105883e-02 -9.95711703e-03 -6.17122324e-03 -6.59692660e-02
 -2.52329316e-02 -5.85594736e-02 -7.48825893e-02  3.29764001e-02
  6.07973896e-03  6.19864948e-02  5.43302903e-03  3.63004506e-02
 -4.46981378e-02  9.28427130e-02  1.64043140e-02  4.35581468e-02
 -3.72492112e-02 -1.27745658e-01 -2.69820280e-02 -7.00748041e-02
 -1.89770944e-02  5.51202940e-03  4.48398553e-02  6.07291125e-02
  2.55929213e-02  2.19874382e-02 -6.06139898e-02 -4.98220660e-02
 -1.77192837e-02 -5.29341027e-02 -4.03532758e-02 -3.24970437e-03
 -1.25286728e-02  8.61169845e-02  4.89204824e-02  2.96234749e-02
  1.41996313e-02  6.53195

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print(sim)

[[0.51327497]]


In [9]:
print(embeddings.shape)

(9, 384)


In [10]:
triples = [
    ("Alice", "works_at", "Google"),
    ("Bob", "works_at", "Amazon"),
    ("Google", "located_in", "California"),
    ("Amazon", "located_in", "Seattle"),
    ("California", "part_of", "USA"),
    ("Seattle", "part_of", "USA"),
    ("Alice", "manages", "Charlie"),
    ("Charlie", "works_on", "Project X"),
    ("Project X", "belongs_to", "Google")
]

In [11]:
import networkx as nx
G=nx.DiGraph()
for subject,relation,obj in triples:
    G.add_edge(subject,obj,relation=relation)

In [12]:
print(G.nodes())

['Alice', 'Google', 'Bob', 'Amazon', 'California', 'Seattle', 'USA', 'Charlie', 'Project X']


In [13]:
print(G.edges(data=True))

[('Alice', 'Google', {'relation': 'works_at'}), ('Alice', 'Charlie', {'relation': 'manages'}), ('Google', 'California', {'relation': 'located_in'}), ('Bob', 'Amazon', {'relation': 'works_at'}), ('Amazon', 'Seattle', {'relation': 'located_in'}), ('California', 'USA', {'relation': 'part_of'}), ('Seattle', 'USA', {'relation': 'part_of'}), ('Charlie', 'Project X', {'relation': 'works_on'}), ('Project X', 'Google', {'relation': 'belongs_to'})]


In [14]:
print(list(G.neighbors("Alice")))

['Google', 'Charlie']


In [15]:
#prime goal id building a traditional RAG then  a Knowledge Graph RAG and then combined two and check the accuracies for all the cases and then decide why the Knowledge graph Rag over Traditional one

documents = [
    "Person B works at Company X",
    "Company X owns Warehouse Y",
    "Warehouse Y stores Chemical Z",
    "Chemical Z is monitored by Security Unit P",
    "Security Unit P operates from Facility F",
    "Facility F is located in Mumbai"
]


In [16]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

In [17]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
print("model loaded successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5368.03it/s]


model loaded successfully


In [18]:
embeddings = model.encode(documents)

In [19]:
print(embeddings.shape)

(6, 384)


In [20]:
print(embeddings.shape)
print(len(embeddings[0]))
print(type(embeddings))

(6, 384)
384
<class 'numpy.ndarray'>


In [21]:
def retrieve(query, top_k=3):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    return [
        documents[i]
        for i in top_indices
    ]

In [27]:
question = "Where is the facility connected to Person B located?"

In [28]:
context = retrieve(question)

for doc in context:
    print(doc)

Person B works at Company X
Security Unit P operates from Facility F
Facility F is located in Mumbai


In [29]:
#as we can see that relevant looking texts are retrieved as per the similarity between the question asked and the document we have so accordingly it has given output
#but for more relevant links and more appropriate answer and most importantly to the point answer we need the Knowledge-Graph for RAG generally used now for LLM's
#building the knowledge graph first

import networkx as nx

G = nx.DiGraph()

In [30]:
G.add_edge(
    "Person B",
    "Company X",
    relation="works_at"
)

G.add_edge(
    "Company X",
    "Warehouse Y",
    relation="owns"
)

G.add_edge(
    "Warehouse Y",
    "Chemical Z",
    relation="stores"
)

G.add_edge(
    "Chemical Z",
    "Security Unit P",
    relation="monitored_by"
)

G.add_edge(
    "Security Unit P",
    "Facility F",
    relation="operates_from"
)

G.add_edge(
    "Facility F",
    "Mumbai",
    relation="located_in"
)

In [33]:
print("Nodes:")
print(G.nodes())

print("\nEdges:")
print(G.edges(data=True))

Nodes:
['Person B', 'Company X', 'Warehouse Y', 'Chemical Z', 'Security Unit P', 'Facility F', 'Mumbai']

Edges:
[('Person B', 'Company X', {'relation': 'works_at'}), ('Company X', 'Warehouse Y', {'relation': 'owns'}), ('Warehouse Y', 'Chemical Z', {'relation': 'stores'}), ('Chemical Z', 'Security Unit P', {'relation': 'monitored_by'}), ('Security Unit P', 'Facility F', {'relation': 'operates_from'}), ('Facility F', 'Mumbai', {'relation': 'located_in'})]


In [36]:
path = nx.shortest_path(
    G,
    source="Person B",
    target="Mumbai"
)

print(path)

['Person B', 'Company X', 'Warehouse Y', 'Chemical Z', 'Security Unit P', 'Facility F', 'Mumbai']


In [37]:
for i in range(len(path)-1):#here path basically represents each segment or list 
    source = path[i]#current node
    target = path[i+1]#the next node
    relation = G[source][target]["relation"]#getting the relation out of the source and target

    print(f"{source} --{relation}--> {target}")

Person B --works_at--> Company X
Company X --owns--> Warehouse Y
Warehouse Y --stores--> Chemical Z
Chemical Z --monitored_by--> Security Unit P
Security Unit P --operates_from--> Facility F
Facility F --located_in--> Mumbai


In [38]:
question = "Where is the facility connected to Person B located?"

In [39]:
current = "Person B"#represents the current position

while True:#infinite loop

    neighbors = list(G.neighbors(current))#finding the neighbours of the current node

    if not neighbors:#no neighbours check
        break

    next_node = neighbors[0]

    relation = G[current][next_node]["relation"]

    if relation == "located_in":#as we know the answer so mentioning it from before hand but normally it will be found out by the llm
        print("Answer:", next_node)
        break

    current = next_node#so here we will be getting the accurate answer and not the paragraph

Answer: Mumbai
